# 07 — Compare experiments

Every finished run in one table, and the comparisons the dissertation reports.

**Responsibility:** compare. Trains nothing, reads only `results/`.

## The rule this notebook enforces

The noise floor on this task is **0.067 macro-AUC** — two byte-identical
configurations differing only in random seed were measured that far apart. Every
comparison below is labelled against it, and a difference inside it is reported
as *no difference detected*, which is a finding rather than a failure.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dataset_config as config
from dataset_config import Config, TASKS

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

from core.experiment import load_experiments

NOISE_FLOOR = 0.067

In [ ]:
runs = pd.DataFrame(load_experiments())
if runs.empty:
    raise SystemExit("no finished runs in results/")
print(f"{len(runs)} finished runs\n")
cols = [c for c in ["run", "pipeline", "task", "model", "seed", "best_epoch",
                    "val_auc", "test_auc", "test_accuracy",
                    "test_balanced_accuracy", "trainable_params"] if c in runs]
print(runs[cols].to_string(index=False))

## Mean over seeds

A single run is never reported on its own.

In [ ]:
key = ["pipeline", "task", "model", "augmentation", "freeze_until"]
key = [k for k in key if k in runs]
grouped = runs.groupby(key).agg(
    n_seeds=("seed", "nunique"),
    test_auc_mean=("test_auc", "mean"), test_auc_std=("test_auc", "std"),
    test_bal_mean=("test_balanced_accuracy", "mean"),
    best_epoch_mean=("best_epoch", "mean")).round(4).sort_values(
        "test_auc_mean", ascending=False)
print(grouped.to_string())
print(f"\nnoise floor: {NOISE_FLOOR} macro-AUC")
print("configurations closer than that are indistinguishable, whatever the ordering.")

## Authors' pipeline against mine

The comparison the thesis exists to make. Note what it is NOT: the two pipelines
use different tasks, so this compares *methods on their own targets*, not two
methods on one target.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colours = {"authors": "#c05621", "mine": "#2b6cb0"}
plot = runs.dropna(subset=["test_auc"]).sort_values("test_auc")
labels = plot.apply(lambda r: f"{r['task']}·{r['model']}", axis=1)
ax.barh(range(len(plot)), plot.test_auc,
        color=[colours.get(p, "grey") for p in plot.pipeline])
ax.set_yticks(range(len(plot)), labels, fontsize=8)
ax.axvline(0.5, color="grey", ls=":", lw=1)
ax.set_xlabel("test macro-AUC (patient level)")
ax.set_title("Every run  ·  orange = authors' pipeline, blue = mine")
for i, (v, b) in enumerate(zip(plot.test_auc, plot.get("test_trivial_baseline_accuracy", plot.test_auc * 0))):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=7)
plt.tight_layout(); plt.show()

## Published reference points

What the authors report, so any number above can be read against something.

| task | their result | measured on |
|---|---|---|
| pCR, overall | AUC 0.72 | 175 patients, pooled |
| pCR, I-SPY2 | AUC 0.78 | 99 patients |
| pCR, I-SPY1 | AUC 0.68 | 35 patients |
| pCR, DUKE | AUC 0.54 | 41 patients |
| HER2 | AUC 0.744 | THDA-ResNet, I-SPY pooled |
| subtype, 3 classes | — | **they never attempted it** |

The often-quoted pCR AUC 0.94 is a 40-patient HR+/HER2− subgroup scored by a
tabular model on clinical variables. Reproduced from their own prediction file:
imaging alone gives 0.8886 on that subgroup, the clinical model 0.9371.

In [ ]:
REFERENCE = {"pcr": 0.72, "her2": 0.744}
for task, target in REFERENCE.items():
    sub = runs[(runs.pipeline == "authors") & (runs.task == task)]
    if sub.empty:
        print(f"{task:<6} target {target:.3f}   — not run yet")
        continue
    got = sub.test_auc.mean()
    delta = got - target
    verdict = ("reproduced" if abs(delta) <= NOISE_FLOOR
               else "ABOVE the noise floor — investigate")
    print(f"{task:<6} target {target:.3f}   ours {got:.4f}   "
          f"delta {delta:+.4f}   {verdict}")

## Export for the dissertation

In [ ]:
out = config.RESULTS_DIR / "all_experiments.csv"
runs.to_csv(out, index=False)
print("written:", out)